# TN1 — ModernTCN ở ngân sách nhỏ

## Câu hỏi

TN1 tới giờ cho một kết quả không thuận với giả thuyết ban đầu: **LSTM hơn cả
hai biến thể TCN**, và thu nhỏ LSTM 96% gần như không mất gì.

| cấu hình | tham số | cv_score | GHIJ |
|---|---|---|---|
| LSTM-352 | 1.502.713 | 0,7570 ± 0,0041 | 0,8103 ± 0,0154 |
| LSTM-67 | 56.908 | 0,7532 ± 0,0020 | 0,8017 ± 0,0025 |
| CNN-LSTM-58 | 55.667 | 0,7527 ± 0,0037 | chưa chạy |
| TCN-64 WeightNorm | 150.745 | 0,7463 ± 0,0030 | chưa chạy |
| TCN-64 BatchNorm | 151.513 | 0,7423 ± 0,0044 | chưa chạy |
| DS-TCN-64 | 56.281 | 0,7421 ± 0,0007 | 0,7958 ± 0,0154 |
| BiLSTM-41 | 57.507 | 0,7398 ± 0,0046 | chưa chạy |

Cả bốn biến thể tích chập đều xếp dưới cả hai biến thể hồi quy. Nhưng cả bốn
đều thuộc **một kiểu thiết kế**: tích chập nhân quả kernel nhỏ, độ giãn tăng
dần, theo Bai et al. 2018.

Câu hỏi của notebook này: thứ thua là *ý tưởng dùng tích chập cho bài toán
này*, hay chỉ là *kiểu thiết kế tích chập năm 2018*?

ModernTCN (Luo, Wang 2024, ICLR) trả lời được, vì nó làm khác hẳn ở hai chỗ:

    Bai 2018      từng mẫu một, kernel 3, độ giãn 1-2-4-8-16-32 để phủ xa
    ModernTCN     chia chuỗi thành đoạn trước, kernel 31 phủ xa ngay một lớp

Đặt ở **56.985 tham số**, lệch 0,14% so với LSTM-67 (56.908) — nên nếu có chênh
lệch điểm thì không phải do bên nào được nhiều tham số hơn.

## Bản cài đặt lấy từ đâu

Từ mã của chính tác giả, nhánh dự báo ngắn hạn:
`ModernTCN-short-term/models/ModernTCN.py`, lớp `Block`,
`ReparamLargeKernelConv` và `forward_feature`.

Không viết theo mô tả trong bài báo. Bốn chi tiết dưới đây chỉ đọc mã mới biết,
và cả bốn đều đủ sức làm hỏng kết luận nếu làm sai:

| chi tiết | đọc mô tả dễ đoán thành | mã tác giả thật sự làm |
|---|---|---|
| số đoạn | 49, nếu tính đệm theo phần dư | **50**, vì đệm là hằng `patch_size − patch_stride` |
| cách đệm | thêm số 0 | **lặp lại giá trị cuối** 4 lần |
| nối tắt | hai cái, như khối Transformer | **một** cái, ôm cả tích chập lẫn FFN |
| BatchNorm | một cái trong FFN | **một cái cho mỗi nhánh kernel**, không có trong FFN |

Chỗ đầu tiên là chỗ nguy hiểm nhất: 49 đoạn thay vì 50 thì model vẫn train
được, vẫn ra điểm, chỉ là sai kiến trúc và không ai biết. Mục 2 kiểm đúng con
số này.

## Một chỗ cố ý bỏ so với mã gốc

`Block.__init__` khai báo đủ bộ `ffn2pw1`, `ffn2act`, `ffn2pw2`, `ffn2drop1`,
`ffn2drop2` — nhưng `forward` **không gọi cái nào**. Chúng nằm trong model và
được `count_params` đếm, mà không tham gia dự báo.

Ở đây bỏ hẳn. Giữ lại thì số tham số báo cáo phồng lên vì phần chết, và so
"cùng ngân sách tham số" với LSTM-67 mất ý nghĩa.

Khối đó vốn để trộn thông tin giữa các **biến**. Chuỗi ở đây chỉ có một biến,
nên dù có gọi cũng không trộn được gì.

## Cấu hình

    độ dài vào / ra        200 / 25              như mọi cấu hình TN1
    bề rộng đoạn           8
    bước dịch đoạn         4                     hai đoạn cạnh nhau chung 4 mẫu
    số đoạn                50
    số kênh                32
    số khối                3
    kernel rộng / hẹp      31 / 5
    FFN mở rộng            32 -> 64 -> 32
    chuẩn hoá              BatchNorm
    dropout                0
    RevIN                  tắt
    tách xu thế/mùa vụ     tắt
    lớp ra                 Linear(50 x 32 = 1600, 25)

RevIN và tách xu thế có trong bài gốc nhưng để tắt ở bản đầu: bật thêm là đổi
thêm biến. RevIN đã có thí nghiệm riêng ở TN2.

**70% tham số nằm ở lớp ra** (40.025 / 56.985), ba khối tích chập chưa tới
17.000. Đó là hệ quả của việc trải phẳng cả 50 đoạn, vốn có trong thiết kế gốc,
không phải lỗi cài đặt.

## Giới hạn của kết luận rút ra được

Cấu hình này do đồ án chọn để vừa ngân sách 56–60k, **không phải cấu hình tác
giả đã chứng minh tốt** trên bài toán nào. Vì vậy kết quả chỉ phát biểu được là
"ModernTCN ở cấu hình nhỏ này cho ra như vậy trên dữ liệu này", không phát biểu
được "ModernTCN thua" hay "ModernTCN thắng".

## 1. Chuẩn bị Colab

Mount Drive để lấy cửa sổ train đã cắt ở `DATA_PREPARE.ipynb`.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Tải mã nguồn rồi vào thư mục đó. Xem dòng `commit đồ án` để chắc đang chạy bản mới.

In [2]:
# Xoá trước để chạy lại ô này luôn lấy mã mới nhất, không dính bản cũ.
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

/content/UWB_RADAR

thư mục làm việc : /content/UWB_RADAR
commit đồ án     : b72df4b
commit MobiVital : 4319731 (đã ghim)
GPU              : Tesla T4, 15360 MiB


Lấy `by_user/` và `windows/` từ Drive. Không cần CSV thô 13 GB.

In [3]:
!python scripts/restore_processed_data_on_drive.py

by_user   : bung /content/drive/MyDrive/mobivital/by_user.tar ...
            12 tệp
windows   : bung /content/drive/MyDrive/mobivital/windows.tar.gz ...
            dev_cv 8 tệp, final_train có

2.5G	data/processed/by_user
503M	data/processed/windows


## 2. Kiểm bản cài đặt

Chín phép kiểm, mất vài giây. Chạy CV mất khoảng hai giờ — sai một chi tiết là
hai giờ đó cho ra kết luận vô nghĩa.

Bốn phép trong số đó nhắm đúng bốn chỗ dễ chệch khỏi mã tác giả:

    mục 5   chia ra đúng 50 đoạn, không phải 49
    mục 6   đệm bằng cách lặp mẫu cuối, không phải đệm 0
    mục 7   xoá trọng số nhánh kernel rộng, rồi nhánh hẹp — đầu ra phải đổi,
            chứng tỏ cả hai thật sự nối vào forward chứ không nằm chết
    mục 8   dựng lại khối bằng tay rồi so: đúng một nối tắt ôm cả khối

Vì sao mục 6 đáng kiểm: đệm 0 sẽ bịa ra một bậc nhảy giả ở **cuối** cửa sổ —
đúng chỗ model cần nhìn kỹ nhất để đoán bước kế tiếp.

Vì sao mục 9 đáng kiểm: nó chặn đúng cái bẫy ConvFFN2 nói ở đầu notebook — có
phần khai báo mà `forward` không gọi, làm phồng số tham số báo cáo.

**Phải gõ đủ `--channels 32 --n_blocks 3`.** Mặc định của hai cờ này là 64 và 6,
tức cấu hình của TCN chứ không phải ModernTCN. Quên cả hai thì ra model 196.313 tham số, quên mỗi `--n_blocks`
thì ra 73.593 — cả hai đều chạy trơn tru mà sai kiến trúc — nên `run_id` được đặt để luôn ghi cả
hai con số vào tên, quên là lộ ra ngay chứ không đè kết quả cũ.

In [4]:
!python scripts/check_model.py --model modern_tcn --channels 32 --n_blocks 3 \
    --compare-with lstm --compare-hidden 67

Kiểm model: modern_tcn

1. Hình dạng vào ra và giá trị
   vào (4, 200)  ->  ra (4, 25)
   ra đúng (4, 25)                                            đạt
   mọi giá trị hữu hạn                                        đạt

2. Gradient
   42/42 tham số nhận được gradient
   mọi tham số đều lan ngược tới                              đạt

3. Số tham số
   56985

4. Lưu và nạp lại state_dict
   nạp lại cho ra đúng đầu ra cũ                              đạt

5. Riêng ModernTCN — chia đoạn
   (4, 1, 200) -> đệm (4, 1, 204) -> đoạn (4, 32, 50)
   đệm 4 = patch_size - patch_stride, hằng số                 đạt
   ra đúng 50 đoạn, không phải 49                             đạt

6. Đệm bằng cách LẶP giá trị cuối, không phải đệm 0
   4 giá trị đệm, lệch lớn nhất so với mẫu cuối: 0.000000
   mọi giá trị đệm bằng đúng mẫu cuối                         đạt
   không phải đệm 0                                           đạt

7. Cả hai nhánh kernel đều nối vào forward
   xoá dw_large làm đổi đầu ra — nhánh nà

## 3. ModernTCN-32 — 4 fold CV, 3 seed

Chín phép kiểm đạt thì mới chạy.

Cấu hình huấn luyện giữ y nguyên như sáu cấu hình trước: 20 epoch, Adam lr 1e-4,
batch 64, MSE, `corr` 0,9, bốn fold cũ. **Chỉ đổi kiến trúc.**

Tên cấu hình là `modern_tcn_c32_n3_mse_corr0.9_seed<N>`. Tên của các lần chạy
trước không đổi — đã đối chiếu từng ký tự cho cả chín `run_id` đang có.

Sau mỗi fold script tự nén rồi chép sang Drive, tên chứa cấu hình nên không đè
tệp nào. Ngắt phiên giữa chừng thì chạy lại ô này, fold đã xong được bỏ qua.

12 lần train. Chưa đo được thời gian thật của kiến trúc này; hai kiến trúc gần
nhất mất 1 đến 1,5 giờ, nên **ước lượng khoảng 1,5 giờ**. Phần chấm điểm chiếm
khoảng 25 phút mỗi seed và không đổi theo kiến trúc.

In [5]:
!python scripts/run_cv.py --experiment tn1 --model modern_tcn --channels 32 --n_blocks 3 --seed 0
!python scripts/run_cv.py --experiment tn1 --model modern_tcn --channels 32 --n_blocks 3 --seed 1
!python scripts/run_cv.py --experiment tn1 --model modern_tcn --channels 32 --n_blocks 3 --seed 2

thực nghiệm tn1  -> runs/tn1/
cấu hình modern_tcn_c32_n3_mse_corr0.9_seed0
thiết bị  Tesla T4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
210964 cửa sổ train
epoch  0  mse 0.03221  pearson 0.4633   0.4 phút
epoch  1  mse 0.02284  pearson 0.5367   0.7 phút
epoch  2  mse 0.02100  pearson 0.5693   1.1 phút
epoch  3  mse 0.01976  pearson 0.5881   1.4 phút
epoch  4  mse 0.01892  pearson 0.6000   1.7 phút
epoch  5  mse 0.01826  pearson 0.6082   2.1 phút
epoch  6  mse 0.01782  pearson 0.6141   2.4 phút
epoch  7  mse 0.01753  pearson 0.6190   2.8 phút
epoch  8  mse 0.01721  pearson 0.6230   3.1 phút
epoch  9  mse 0.01701  pearson 0.6255   3.5 phút
epoch 10  mse 0.01677  pearson 0.6285   3.8 phút
epoch 11  mse 0.01662  pearson 0.6306   4.2 phút
epoch 12  mse 0.01650  pearson 0.6323   4.5 phút
epoch 13  mse 0.01633  pearson 0.6340   4.9 phút
epoch 14  mse 0.01617  pearson 0.6357   5.2 phút
epoch 15  mse 0.01607  pearson 0.6368   5.6 phút
epoch 16  ms

## 4. Cất kết quả

`run_cv.py` đã tự nén sau mỗi fold. Ô dưới nén lại một lần sau khi xong cả 12
lần chạy, ra tên riêng `tn1_modern_tcn.zip` để không đè tệp nén của notebook
nào khác.

Bảng so đủ các cấu hình dựng ở `TN1_final_evaluation.ipynb`, ô gộp ở đó tự quét
mọi tệp `tn1*.zip` trên Drive nên tự nhận tệp này.

In [6]:
!python scripts/save_results.py tn1 --out tn1_modern_tcn

runs/tn1/  ->  runs/tn1_modern_tcn.zip   (2.7 MB)
   15 dòng metric trong summary.csv

Bên trong:
        0  2026-09-06 22:55   tn1/
        0  2026-09-06 20:43   tn1/modern_tcn_c32_n3_mse_corr0.9_seed0_val_AB/
        0  2026-09-06 20:55   tn1/modern_tcn_c32_n3_mse_corr0.9_seed0_val_CE/
        0  2026-09-06 21:07   tn1/modern_tcn_c32_n3_mse_corr0.9_seed0_val_DF/
        0  2026-09-06 21:18   tn1/modern_tcn_c32_n3_mse_corr0.9_seed0_val_KL/
        0  2026-09-06 21:29   tn1/modern_tcn_c32_n3_mse_corr0.9_seed1_val_AB/
        0  2026-09-06 21:41   tn1/modern_tcn_c32_n3_mse_corr0.9_seed1_val_CE/
        0  2026-09-06 21:54   tn1/modern_tcn_c32_n3_mse_corr0.9_seed1_val_DF/
        0  2026-09-06 22:05   tn1/modern_tcn_c32_n3_mse_corr0.9_seed1_val_KL/
        0  2026-09-06 22:15   tn1/modern_tcn_c32_n3_mse_corr0.9_seed2_val_AB/
        0  2026-09-06 22:28   tn1/modern_tcn_c32_n3_mse_corr0.9_seed2_val_CE/
        0  2026-09-06 22:40   tn1/modern_tcn_c32_n3_mse_corr0.9_seed2_val_DF/
        0

## 5. Bảng so trong phiên này

Ô riêng, không gộp vào ô train: muốn xem lại bảng thì chạy lại vài giây, chứ
không phải train lại 12 fold.

Chỉ thấy các cấu hình đã có trong `runs/tn1/` của phiên này. Bảng đầy đủ vẫn là
`TN1_final_evaluation.ipynb`.

In [7]:
!python scripts/compare_cv.py --experiment tn1


BẢNG 1 — cv_score, thực nghiệm tn1
cấu hình                         tham số  seed    cv_mean  seed_std  fold_std   từng seed
--------------------------------------------------------------------------------------------------------------
modern_tcn_c32_n3_mse_corr0.9      56985     3   0.740526  0.004477  0.074918   s0 0.7438  s1 0.7354  s2 0.7424

cv_mean  = trung bình cv_score của các seed. cv_score của một seed là
           trung bình điểm macro 4 fold; macro = trung bình theo NGƯỜI.
seed_std = dao động giữa các seed. Chênh lệch giữa hai cấu hình nhỏ hơn
           số này thì chưa kết luận được.
fold_std = dao động giữa 4 fold, trung bình trên các seed. Nói dữ liệu
           giữa các người khác nhau ra sao, KHÔNG dùng để so cấu hình.



## 6. Ngắt phiên

Colab giữ runtime sau khi ô cuối chạy xong và vẫn tính giờ. Ô này đóng phiên
lại. Kết quả đã nén sang Drive ở mục 4 nên ngắt ở đây không mất gì.

Bấm liên tiếp các ô mục 3, 4, 5, 6 thì Colab xếp hàng chạy lần lượt, không cần
ngồi canh.

In [ ]:
from google.colab import runtime
runtime.unassign()